In [ ]:
!nvidia-smi

Wed Oct  7 12:42:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
DATA_DIR = "/content/drive/MyDrive/greek-news-nlp"
print(os.listdir(DATA_DIR))

Mounted at /content/drive
['el_train.jsonl', 'el_val.jsonl', 'el_test.jsonl']


In [3]:
!pip install -q -U transformers accelerate

import unicodedata
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

DATA_DIR = "/content/drive/MyDrive/greek-news-nlp"
MODEL_NAME = "nlpaueb/bert-base-greek-uncased-v1"
LABEL_COL = "GPT-IPTC-label"
MAX_LEN = 256


def prep(text):
    # Το GreekBERT uncased θέλει πεζά και χωρίς τόνους
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.lower()


train_df = pd.read_json(f"{DATA_DIR}/el_train.jsonl", lines=True)
val_df = pd.read_json(f"{DATA_DIR}/el_val.jsonl", lines=True)

labels = sorted(train_df[LABEL_COL].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.enc = tokenizer(
            [prep(t) for t in df["text_clean"]], truncation=True, max_length=MAX_LEN
        )
        self.labels = [label2id[l] for l in df[LABEL_COL]]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


train_ds = NewsDataset(train_df)
val_ds = NewsDataset(val_df)
print(len(train_ds), len(val_ds), len(labels))
print(tokenizer.convert_ids_to_tokens(train_ds[0]["input_ids"])[:15])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 80.5 MB/s eta 0:00:00


config.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/530k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

3675 787 17
['[CLS]', '"', 'αλ', 'τσαντ', '##ιρ', '##ι', '"', ':', 'πτωση', 'στα', 'νουμερα', 'τηλεθεαση', '##ς', 'για', 'τον']


In [4]:
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
)

# Ίδια λογική με το baseline: περισσότερο βάρος στις μικρές κατηγορίες
weights = compute_class_weight(
    "balanced", classes=np.arange(len(labels)), y=np.array(train_ds.labels)
)
class_weights = torch.tensor(weights, dtype=torch.float)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        y = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fn(outputs.logits, y)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro"),
    }


args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  454MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  454MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=920, training_loss=0.6828544782555621, metrics={'train_runtime': 247.96, 'train_samples_per_second': 59.284, 'train_steps_per_second': 3.71, 'total_flos': 1934126707968000.0, 'train_loss': 0.6828544782555621, 'epoch': 4.0})

In [5]:
from sklearn.metrics import classification_report

out = trainer.predict(val_ds)
val_pred = out.predictions.argmax(axis=-1)
print(classification_report(out.label_ids, val_pred, target_names=labels, zero_division=0))

SAVE_DIR = f"{DATA_DIR}/greek-bert-news-classifier"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

                                           precision    recall  f1-score   support

   arts, culture, entertainment and media       0.84      0.87      0.86        79
                  conflict, war and peace       0.62      0.69      0.65        26
                   crime, law and justice       0.88      0.79      0.83        63
disaster, accident and emergency incident       0.78      0.78      0.78        36
            economy, business and finance       0.92      0.68      0.78        99
                                education       0.75      0.87      0.81        31
                              environment       0.83      0.83      0.83        18
                                   health       0.75      0.85      0.80        39
                           human interest       0.71      0.65      0.68        62
                                   labour       0.53      1.00      0.69        21
                    lifestyle and leisure       0.35      0.75      0.48         8
   

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/drive/MyDrive/greek-news-nlp/greek-bert-news-classifier


In [6]:
test_df = pd.read_json(f"{DATA_DIR}/el_test.jsonl", lines=True)
test_ds = NewsDataset(test_df)

out = trainer.predict(test_ds)
test_pred = out.predictions.argmax(axis=-1)
y_true = out.label_ids

print("Accuracy:", round(accuracy_score(y_true, test_pred), 3))
print("Macro-F1:", round(f1_score(y_true, test_pred, average="macro"), 3))
print()
print(classification_report(y_true, test_pred, target_names=labels, zero_division=0))

Accuracy: 0.825
Macro-F1: 0.77

                                           precision    recall  f1-score   support

   arts, culture, entertainment and media       0.91      0.88      0.89        80
                  conflict, war and peace       0.62      0.58      0.60        26
                   crime, law and justice       0.88      0.81      0.84        63
disaster, accident and emergency incident       0.76      0.86      0.81        37
            economy, business and finance       0.85      0.76      0.80       100
                                education       0.83      0.78      0.81        32
                              environment       0.71      0.88      0.79        17
                                   health       0.74      0.90      0.81        39
                           human interest       0.82      0.76      0.79        62
                                   labour       0.56      1.00      0.71        20
                    lifestyle and leisure       0.28  